## Part 2.1: Initialise Profiler Run State

### Objective

Initialise the shared runtime state required by the CSV schema diagnostic profiler before loading type rules, schema registries, or source files.

Each profiler execution receives one unique session ID. The same session ID is used throughout Parts 2.1–2.6 so that all audit events, file observations, schema records, and final summary information can be traced to one profiler run.

### Dependencies

This section depends on configuration and runtime objects established earlier in the notebook:

* `MELBOURNE_TIMEZONE` from Global Runtime Settings;
* `RAW_DATA_PATH` from Part 1.2;
* `LOG_AUDIT_PATH` from Part 1.2;
* `MASTER_SCHEMA_PATH` from Part 1.2;
* `PY_SCHEMA_PATH` from Part 1.2;
* `TYPE_RULES_PATH` from Part 1.2;
* `PIPELINE_AUDIT_PATH` from Part 1.2;
* `PROFILE_SAMPLE_ROWS` from Part 1.2.

This section does not reconstruct paths or reload project configuration.

### Run State

The profiler initialises:

* `SESSION_ID` to identify the current profiler execution;
* `PROFILER_START_TIME` using Melbourne local time;
* warning and error collections;
* counters for discovered, successful, failed, and structurally different files;
* collections for file-level observations, failures, and schema comparison details.

The session ID identifies an execution only. It is separate from the approved schema version.

### Central Audit Logging

A shared `log_event()` function is created for all profiler sections.

Each log entry contains:

* a timezone-aware Melbourne timestamp;
* the profiler session ID;
* a severity level;
* an event message.

Supported severity levels are:

* `INFO`
* `WARNING`
* `ERROR`
* `CRITICAL`

The function appends every event to `pipeline_audit.log`, prints the event in the notebook, and records warnings and errors in the current run state.

### Prerequisite Validation

Before the profiler begins, this section confirms that:

* all required path variables are available;
* `RAW_DATA_PATH` exists and is a directory;
* `LOG_AUDIT_PATH` exists and is a directory;
* `PIPELINE_AUDIT_PATH` exists and is a file;
* `PROFILE_SAMPLE_ROWS` contains a valid profiling policy.

Missing global prerequisites stop execution before any schema files or CSV files are processed.

### Expected Outcome

* A unique profiler session ID is generated.
* Melbourne profiler start time is recorded.
* Shared counters and result collections are initialised.
* Central audit logging is ready for Parts 2.2–2.6.
* Profiler prerequisites are verified.
* A `SESSION_START` event is appended to `pipeline_audit.log`.


In [ ]:
# ==========================================
# Part 2.1: Initialise Profiler Run State
# ==========================================

from datetime import datetime
from pathlib import Path
from uuid import uuid4


# ------------------------------------------
# Validate dependencies from earlier parts
# ------------------------------------------

REQUIRED_RUNTIME_VARIABLES = {
    "MELBOURNE_TIMEZONE": MELBOURNE_TIMEZONE,
    "RAW_DATA_PATH": RAW_DATA_PATH,
    "LOG_AUDIT_PATH": LOG_AUDIT_PATH,
    "MASTER_SCHEMA_PATH": MASTER_SCHEMA_PATH,
    "PY_SCHEMA_PATH": PY_SCHEMA_PATH,
    "TYPE_RULES_PATH": TYPE_RULES_PATH,
    "PIPELINE_AUDIT_PATH": PIPELINE_AUDIT_PATH,
    "PROFILE_SAMPLE_ROWS": PROFILE_SAMPLE_ROWS,
}


for variable_name, variable_value in REQUIRED_RUNTIME_VARIABLES.items():
    if variable_value is None and variable_name != "PROFILE_SAMPLE_ROWS":
        raise RuntimeError(
            f"Required runtime variable is unavailable: {variable_name}"
        )


# ------------------------------------------
# Validate resolved path object types
# ------------------------------------------

REQUIRED_PATH_VARIABLES = {
    "RAW_DATA_PATH": RAW_DATA_PATH,
    "LOG_AUDIT_PATH": LOG_AUDIT_PATH,
    "MASTER_SCHEMA_PATH": MASTER_SCHEMA_PATH,
    "PY_SCHEMA_PATH": PY_SCHEMA_PATH,
    "TYPE_RULES_PATH": TYPE_RULES_PATH,
    "PIPELINE_AUDIT_PATH": PIPELINE_AUDIT_PATH,
}


for variable_name, path_value in REQUIRED_PATH_VARIABLES.items():
    if not isinstance(path_value, Path):
        raise TypeError(
            f"{variable_name} must be a pathlib.Path object."
        )


# ------------------------------------------
# Validate profiler prerequisites
# ------------------------------------------

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"RAW_DATA_PATH does not exist:\n{RAW_DATA_PATH}"
    )

if not RAW_DATA_PATH.is_dir():
    raise NotADirectoryError(
        f"RAW_DATA_PATH is not a directory:\n{RAW_DATA_PATH}"
    )


if not LOG_AUDIT_PATH.exists():
    raise FileNotFoundError(
        f"LOG_AUDIT_PATH does not exist:\n{LOG_AUDIT_PATH}"
    )

if not LOG_AUDIT_PATH.is_dir():
    raise NotADirectoryError(
        f"LOG_AUDIT_PATH is not a directory:\n{LOG_AUDIT_PATH}"
    )


if not PIPELINE_AUDIT_PATH.exists():
    raise FileNotFoundError(
        f"PIPELINE_AUDIT_PATH does not exist:\n"
        f"{PIPELINE_AUDIT_PATH}"
    )

if not PIPELINE_AUDIT_PATH.is_file():
    raise FileExistsError(
        f"PIPELINE_AUDIT_PATH is not a file:\n"
        f"{PIPELINE_AUDIT_PATH}"
    )


if (
    PROFILE_SAMPLE_ROWS is not None
    and not isinstance(PROFILE_SAMPLE_ROWS, int)
):
    raise TypeError(
        "PROFILE_SAMPLE_ROWS must be an integer or null."
    )

if (
    isinstance(PROFILE_SAMPLE_ROWS, int)
    and PROFILE_SAMPLE_ROWS < 0
):
    raise ValueError(
        "PROFILE_SAMPLE_ROWS cannot be negative."
    )


# ------------------------------------------
# Initialise profiler execution identity
# ------------------------------------------

SESSION_ID = uuid4().hex[:8]

PROFILER_START_TIME = datetime.now(
    MELBOURNE_TIMEZONE
)


# ------------------------------------------
# Determine profiling policy
# ------------------------------------------

if PROFILE_SAMPLE_ROWS in (None, 0):
    PROFILING_MODE = "FULL_FILE"
    EFFECTIVE_SAMPLE_ROWS = None
else:
    PROFILING_MODE = "SAMPLED"
    EFFECTIVE_SAMPLE_ROWS = PROFILE_SAMPLE_ROWS


# ------------------------------------------
# Initialise run-level event collections
# ------------------------------------------

run_warnings = []
run_errors = []


# ------------------------------------------
# Initialise file-processing counters
# ------------------------------------------

files_discovered_count = 0
files_processed_successfully_count = 0
files_failed_count = 0
files_with_drift_count = 0
files_with_proposed_schema_variance_count = 0


# ------------------------------------------
# Initialise file-level result collections
# ------------------------------------------

discovered_files = []
processed_files = []
failed_files = []

observed_pandas_schemas_by_file = {}
observed_sql_schemas_by_file = {}

drift_details_by_file = {}
proposed_schema_variance_by_file = {}


# ------------------------------------------
# Define central profiler audit logger
# ------------------------------------------

SUPPORTED_LOG_LEVELS = {
    "INFO",
    "WARNING",
    "ERROR",
    "CRITICAL",
}


def log_event(
    message: str,
    level: str = "INFO",
) -> str:
    """
    Append one profiler event to the pipeline audit log,
    print it in the notebook, and update run-level event
    collections when the event is a warning or error.
    """
    normalised_level = level.upper().strip()

    if normalised_level not in SUPPORTED_LOG_LEVELS:
        raise ValueError(
            f"Unsupported log level: {level}. "
            "Expected INFO, WARNING, ERROR, or CRITICAL."
        )

    if not isinstance(message, str) or not message.strip():
        raise ValueError(
            "Audit event message must be a non-empty string."
        )

    timestamp = datetime.now(
        MELBOURNE_TIMEZONE
    ).strftime("%Y-%m-%d %H:%M:%S %Z")

    log_entry = (
        f"[{timestamp}] "
        f"[RUN:{SESSION_ID}] "
        f"[{normalised_level}] "
        f"{message.strip()}"
    )

    # Append rather than overwrite so previous profiler
    # executions remain available for audit review.
    with PIPELINE_AUDIT_PATH.open(
        mode="a",
        encoding="utf-8",
    ) as audit_file:
        audit_file.write(log_entry + "\n")

    print(log_entry)

    event_record = {
        "timestamp": timestamp,
        "session_id": SESSION_ID,
        "level": normalised_level,
        "message": message.strip(),
    }

    if normalised_level == "WARNING":
        run_warnings.append(event_record)

    elif normalised_level in {"ERROR", "CRITICAL"}:
        run_errors.append(event_record)

    return log_entry


# ------------------------------------------
# Record profiler session start
# ------------------------------------------

log_event(
    message=(
        "SESSION_START: CSV schema diagnostic profiler initialised. "
        f"Profiling mode={PROFILING_MODE}; "
        f"sample_rows={EFFECTIVE_SAMPLE_ROWS}."
    ),
    level="INFO",
)


# ------------------------------------------
# Display initialisation summary
# ------------------------------------------

print("\n========== Profiler Run Initialisation ==========\n")
print(f"Session ID       : {SESSION_ID}")
print(
    "Start time       : "
    f"{PROFILER_START_TIME.isoformat(timespec='seconds')}"
)
print(f"Timezone         : {PROFILER_START_TIME.tzname()}")
print(f"Profiling mode   : {PROFILING_MODE}")
print(f"Sample rows      : {EFFECTIVE_SAMPLE_ROWS}")
print(f"Raw data path    : {RAW_DATA_PATH}")
print(f"Audit log path   : {PIPELINE_AUDIT_PATH}")

print("\n" + "=" * 55)
print("✓ Profiler run state initialised.")
print("✓ Runtime prerequisites verified.")
print("✓ Central audit logging is ready.")
print("=" * 55)

## Part 2.2: Create, Load, and Validate Type Rules

### Objective

Prepare the type rules used by the profiler to preserve important CSV columns, translate observed Pandas data types into SQL-compatible types, and apply controlled column-specific SQL overrides.

The rules are stored in `type_rules.json`, whose path was resolved earlier as `TYPE_RULES_PATH`.

### Type-Rule Structure

The rule file contains three sections:

```text
default_pandas_to_sql_mapping
column_name_to_sql_type_overrides
read_csv_explicit_dtypes
```

#### Pandas-to-SQL mappings

These rules translate observed Pandas data types into SQL-compatible definitions.

Examples include:

```text
int64           → INTEGER
float64         → NUMERIC(10, 6)
bool            → BOOLEAN
datetime64[ns]  → TIMESTAMP
object          → VARCHAR(255)
```

#### Column-specific SQL overrides

Some columns require explicit database definitions regardless of their inferred Pandas type.

For example, latitude and longitude fields use controlled numeric precision:

```text
start_lat → NUMERIC(10, 6)
start_lng → NUMERIC(10, 6)
end_lat   → NUMERIC(10, 6)
end_lng   → NUMERIC(10, 6)
```

Overrides use exact column-name matching. Broad substring matching is avoided because it could accidentally change unrelated columns.

#### Explicit CSV read dtypes

Known identifier and text columns are explicitly read as strings to reduce incorrect numeric inference and preserve values such as leading zeros.

Current explicit string columns include:

```text
start_station_name
start_station_id
end_station_name
end_station_id
```

### File-Management Policy

This section applies the following rules:

* If `type_rules.json` does not exist, create it once using the default rules.
* If the file already exists and is valid, preserve its existing values.
* If a required section is missing, add only the missing section from the defaults.
* If the JSON is invalid, preserve the existing file and use the in-memory defaults for the current run.
* If the top-level structure or a required section is malformed, preserve the file and use a safe in-memory replacement for the affected section.
* Never overwrite a valid existing rule file during normal execution.

When missing sections are added, the updated JSON is written atomically to reduce the risk of file corruption during an interrupted write.

### Safe Dtype Resolution

JSON stores dtype names as text, such as `"str"` or `"string"`.

The profiler converts these values through a controlled mapping rather than using `eval()`.

Supported dtype names include:

```text
str
object
int
float
bool
string
```

Unsupported dtype names generate a warning and use a safe `object` fallback for the current run.

### Expected Outcome

* `type_rules.json` exists or safe in-memory defaults are available.
* Existing valid rule values are preserved.
* Missing required sections are restored without replacing unrelated content.
* Explicit CSV dtypes are converted safely into Pandas-compatible dtype definitions.
* SQL mappings and column overrides are ready for later profiling stages.
* All repairs and fallbacks are recorded through the central profiler audit log.


In [ ]:
# ==========================================
# Part 2.2: Create, Load, and Validate
# Type Rules
# ==========================================

import json
from copy import deepcopy
from pathlib import Path

import pandas as pd


# ------------------------------------------
# Define default profiler type rules
# ------------------------------------------

DEFAULT_TYPE_RULES = {
    "default_pandas_to_sql_mapping": {
        "int64": "INTEGER",
        "Int64": "INTEGER",
        "float64": "NUMERIC(10, 6)",
        "bool": "BOOLEAN",
        "boolean": "BOOLEAN",
        "datetime64[ns]": "TIMESTAMP",
        "datetime64[ns, UTC]": "TIMESTAMP",
        "object": "VARCHAR(255)",
        "string": "VARCHAR(255)",
        "category": "VARCHAR(255)",
    },
    "column_name_to_sql_type_overrides": {
        "start_lat": "NUMERIC(10, 6)",
        "start_lng": "NUMERIC(10, 6)",
        "end_lat": "NUMERIC(10, 6)",
        "end_lng": "NUMERIC(10, 6)",
    },
    "read_csv_explicit_dtypes": {
        "start_station_name": "str",
        "start_station_id": "str",
        "end_station_name": "str",
        "end_station_id": "str",
    },
}


REQUIRED_TYPE_RULE_SECTIONS = {
    "default_pandas_to_sql_mapping",
    "column_name_to_sql_type_overrides",
    "read_csv_explicit_dtypes",
}


# ------------------------------------------
# Define controlled dtype-name mapping
# ------------------------------------------

SUPPORTED_DTYPE_NAMES = {
    "str": str,
    "object": object,
    "int": int,
    "float": float,
    "bool": bool,
    "string": pd.StringDtype(),
}


# ------------------------------------------
# Save JSON atomically
# ------------------------------------------

def save_json_atomically(
    file_path: Path,
    data: dict,
) -> None:
    """
    Write JSON to a temporary file and replace the target only
    after the write completes successfully.
    """
    temporary_path = file_path.with_suffix(
        file_path.suffix + ".tmp"
    )

    try:
        with temporary_path.open(
            mode="w",
            encoding="utf-8",
        ) as temporary_file:
            json.dump(
                data,
                temporary_file,
                indent=4,
                ensure_ascii=False,
            )

            temporary_file.write("\n")

        temporary_path.replace(file_path)

    finally:
        if temporary_path.exists():
            temporary_path.unlink()


# ------------------------------------------
# Create default type rules if missing
# ------------------------------------------

def create_default_type_rules_if_missing(
    file_path: Path,
) -> bool:
    """
    Create type_rules.json only when it does not already exist.

    Returns True when the file is created and False when an
    existing file is preserved.
    """
    if file_path.exists():

        if not file_path.is_file():
            raise FileExistsError(
                "TYPE_RULES_PATH exists but is not a file:\n"
                f"{file_path}"
            )

        log_event(
            message=(
                "TYPE_RULES_FOUND: Existing type_rules.json "
                "will be preserved."
            ),
            level="INFO",
        )

        return False

    save_json_atomically(
        file_path=file_path,
        data=DEFAULT_TYPE_RULES,
    )

    log_event(
        message=(
            "TYPE_RULES_CREATED: type_rules.json was created "
            "using the default profiler rules."
        ),
        level="INFO",
    )

    return True


# ------------------------------------------
# Load and validate type rules
# ------------------------------------------

def load_type_rules(
    file_path: Path,
) -> dict:
    """
    Load type_rules.json conservatively.

    Invalid JSON or an invalid top-level structure is preserved
    on disk while the profiler uses in-memory defaults.
    """
    try:
        with file_path.open(
            mode="r",
            encoding="utf-8",
        ) as rule_file:
            loaded_rules = json.load(rule_file)

    except json.JSONDecodeError as error:
        log_event(
            message=(
                "TYPE_RULES_INVALID_JSON: "
                f"{file_path.name} could not be parsed. "
                "The existing file was preserved and in-memory "
                f"defaults will be used. Error={error}"
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_TYPE_RULES)

    except OSError as error:
        log_event(
            message=(
                "TYPE_RULES_READ_FAILED: "
                f"{file_path.name} could not be read. "
                "In-memory defaults will be used. "
                f"Error={error}"
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_TYPE_RULES)

    if not isinstance(loaded_rules, dict):
        log_event(
            message=(
                "TYPE_RULES_INVALID_STRUCTURE: "
                f"{file_path.name} must contain a JSON object. "
                "The existing file was preserved and in-memory "
                "defaults will be used."
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_TYPE_RULES)

    validated_rules = deepcopy(loaded_rules)
    missing_sections = []
    invalid_sections = []

    for section_name in REQUIRED_TYPE_RULE_SECTIONS:

        if section_name not in validated_rules:
            validated_rules[section_name] = deepcopy(
                DEFAULT_TYPE_RULES[section_name]
            )

            missing_sections.append(section_name)

        elif not isinstance(
            validated_rules[section_name],
            dict,
        ):
            validated_rules[section_name] = deepcopy(
                DEFAULT_TYPE_RULES[section_name]
            )

            invalid_sections.append(section_name)

    # Missing sections represent incomplete but repairable JSON.
    # Add only those sections and preserve all existing content.
    if missing_sections:
        save_json_atomically(
            file_path=file_path,
            data=validated_rules,
        )

        log_event(
            message=(
                "TYPE_RULES_REPAIRED: Missing section(s) added "
                "from defaults: "
                + ", ".join(sorted(missing_sections))
            ),
            level="WARNING",
        )

    # Invalid section types are replaced only in memory.
    # The original file remains untouched for human review.
    if invalid_sections:
        log_event(
            message=(
                "TYPE_RULES_INVALID_SECTIONS: The following "
                "section(s) were not JSON objects and were "
                "replaced with defaults in memory only: "
                + ", ".join(sorted(invalid_sections))
            ),
            level="CRITICAL",
        )

    if not missing_sections and not invalid_sections:
        log_event(
            message=(
                "TYPE_RULES_LOADED: Existing type rules passed "
                "structural validation."
            ),
            level="INFO",
        )

    return validated_rules


# ------------------------------------------
# Resolve explicit CSV dtypes safely
# ------------------------------------------

def resolve_explicit_dtypes(
    configured_dtypes: dict,
) -> dict:
    """
    Convert configured dtype names into controlled Python or
    Pandas dtype definitions without using eval().
    """
    resolved_dtypes = {}

    for column_name, configured_dtype in configured_dtypes.items():

        if not isinstance(column_name, str) or not column_name.strip():
            log_event(
                message=(
                    "TYPE_RULE_INVALID_COLUMN: An explicit dtype "
                    "rule contained an invalid column name and was "
                    "ignored."
                ),
                level="WARNING",
            )

            continue

        if not isinstance(configured_dtype, str):
            log_event(
                message=(
                    "TYPE_RULE_INVALID_DTYPE: "
                    f"Column={column_name}; configured dtype must "
                    "be a string. Safe object fallback applied."
                ),
                level="WARNING",
            )

            resolved_dtypes[column_name] = object
            continue

        normalised_dtype = configured_dtype.strip()

        if normalised_dtype in SUPPORTED_DTYPE_NAMES:
            resolved_dtypes[column_name] = (
                SUPPORTED_DTYPE_NAMES[normalised_dtype]
            )

        else:
            log_event(
                message=(
                    "TYPE_RULE_UNSUPPORTED_DTYPE: "
                    f"Column={column_name}; "
                    f"dtype={configured_dtype}. "
                    "Safe object fallback applied."
                ),
                level="WARNING",
            )

            resolved_dtypes[column_name] = object

    return resolved_dtypes


# ------------------------------------------
# Initialise type rules
# ------------------------------------------

create_default_type_rules_if_missing(
    file_path=TYPE_RULES_PATH,
)

TYPE_RULES = load_type_rules(
    file_path=TYPE_RULES_PATH,
)


# ------------------------------------------
# Expose validated rule collections
# ------------------------------------------

PANDAS_TO_SQL_TYPE_MAP = TYPE_RULES[
    "default_pandas_to_sql_mapping"
]

COLUMN_SQL_TYPE_OVERRIDES = TYPE_RULES[
    "column_name_to_sql_type_overrides"
]

EXPLICIT_READ_DTYPES = resolve_explicit_dtypes(
    TYPE_RULES["read_csv_explicit_dtypes"]
)


# ------------------------------------------
# Display type-rule summary
# ------------------------------------------

print("\n========== Type Rule Initialisation ==========\n")

print(f"Type rules file       : {TYPE_RULES_PATH}")
print(
    "Pandas-to-SQL rules  : "
    f"{len(PANDAS_TO_SQL_TYPE_MAP)}"
)
print(
    "SQL column overrides : "
    f"{len(COLUMN_SQL_TYPE_OVERRIDES)}"
)
print(
    "Explicit CSV dtypes  : "
    f"{len(EXPLICIT_READ_DTYPES)}"
)

print("\nExplicit CSV dtype rules:")

for column_name, dtype_value in EXPLICIT_READ_DTYPES.items():
    print(f"  {column_name:<25}: {dtype_value}")

print("\n" + "=" * 55)
print("✓ Type rules created or loaded.")
print("✓ Required rule sections validated.")
print("✓ Explicit CSV dtypes resolved safely.")
print("=" * 55)

## Part 2.3: Initialise, Migrate, and Validate Schema Registries

### Objective

Create or load the persistent schema registries used by the diagnostic profiler, migrate older registry structures when safe, and validate that the registries are ready for later schema analysis.

The profiler uses two separate registries:

* `master_schema.json` manages approved and proposed SQL-compatible schemas.
* `py_schema.json` stores observed Pandas schemas by source filename.

This section prepares registry storage only. It does not discover CSV files, infer schemas, approve proposals, or modify column-level schema definitions.

### Master SQL Schema Registry

The default structure of `master_schema.json` is:

```json
{
  "approval_workflow": {
    "review_required": true,
    "instructions": [
      "Review proposed_schema against the source-data requirements.",
      "Confirm all column names and SQL data types are correct.",
      "Copy proposed_schema into active_schema after approval.",
      "Set active_schema_status to APPROVED.",
      "Assign a new active_schema_version.",
      "Update proposed_schema_metadata.status to APPROVED.",
      "Append an approval record to revision_history."
    ],
    "allowed_active_schema_statuses": [
      "NOT_APPROVED",
      "APPROVED"
    ],
    "approval_note": "The profiler never approves schemas automatically. Approval requires human review."
  },
  "active_schema": {},
  "active_schema_version": null,
  "active_schema_status": "NOT_APPROVED",
  "proposed_schema": {},
  "proposed_schema_metadata": null,
  "revision_history": [],
  "run_history": []
}
```

The registry sections have different responsibilities:

* `approval_workflow` provides embedded human-review instructions.
* `active_schema` stores the approved SQL schema.
* `active_schema_version` identifies the approved schema revision.
* `active_schema_status` indicates whether an approved schema is available.
* `proposed_schema` stores a discovery-generated schema awaiting review.
* `proposed_schema_metadata` records how and when the proposal was created.
* `revision_history` records approved schema revisions.
* `run_history` records profiler execution results.

The profiler must never promote `proposed_schema` into `active_schema` automatically.

### Human Approval Workflow

When a proposed schema has been reviewed and accepted, the reviewer must:

1. Confirm that all proposed column names and SQL types meet the project requirements.
2. Copy the approved schema from `proposed_schema` into `active_schema`.
3. Set `active_schema_status` to `APPROVED`.
4. Assign an `active_schema_version`, such as `1.0.0`.
5. Update `proposed_schema_metadata.status` to `APPROVED`.
6. Add approval metadata, including the approval timestamp and reviewer.
7. Append the approved schema to `revision_history`.

The embedded instructions are valid JSON fields. Standard JSON comments are not used because they would make the registry invalid.

### Observed Pandas Schema Registry

The default structure of `py_schema.json` is:

```json
{
  "schemas_by_file": {},
  "run_history": []
}
```

Later profiler stages will store each source file using:

```json
{
  "latest_observation": {},
  "observation_history": []
}
```

### Registry Migration Policy

Existing registry files may have been created before new fields were introduced.

This section safely migrates valid older registries by:

* adding missing top-level sections from the default structure;
* adding missing fields inside `approval_workflow`;
* preserving existing valid registry values;
* preserving existing history;
* writing safe structural migrations atomically.

If an existing section has an invalid data type:

* the original file is not silently replaced;
* a safe default is used in memory for the current run;
* a critical event is recorded for human review.

Invalid JSON files are preserved and are not overwritten automatically.

### Validation Rules

The section validates that:

* registry files contain JSON objects;
* required top-level sections exist;
* required sections use the expected data types;
* `active_schema_status` uses an allowed value;
* an approved status is not paired with an empty active schema;
* an active schema has an associated version;
* observed file records use the expected object and list structures.

Structural validation is separate from schema normalization. This section does not correct historical column types.

### Expected Outcome

* Missing registry files are created with valid default structures.
* Existing valid registry data and history are preserved.
* Older registries receive safe structural migrations.
* `approval_workflow` is present in `master_schema.json`.
* Missing approval instructions are added automatically.
* Invalid registry content is reported without being silently destroyed.
* `master_registry` and `py_registry` are available in memory for Parts 2.4–2.7.


In [ ]:
# ==========================================
# Part 2.3: Initialise, Migrate, and Validate
# Schema Registries
# ==========================================

import json
from copy import deepcopy
from pathlib import Path
from typing import Any


# ------------------------------------------
# Validate required dependencies
# ------------------------------------------

REQUIRED_PART_2_3_VARIABLES = {
    "MASTER_SCHEMA_PATH": MASTER_SCHEMA_PATH,
    "PY_SCHEMA_PATH": PY_SCHEMA_PATH,
    "LOG_AUDIT_PATH": LOG_AUDIT_PATH,
}

for variable_name, variable_value in (
    REQUIRED_PART_2_3_VARIABLES.items()
):
    if not isinstance(variable_value, Path):
        raise TypeError(
            f"{variable_name} must be a pathlib.Path object."
        )


if not LOG_AUDIT_PATH.exists():
    LOG_AUDIT_PATH.mkdir(
        parents=True,
        exist_ok=True,
    )

    log_event(
        message=(
            "LOG_AUDIT_PATH_CREATED: "
            f"path={LOG_AUDIT_PATH}."
        ),
        level="INFO",
    )


if not LOG_AUDIT_PATH.is_dir():
    raise NotADirectoryError(
        f"LOG_AUDIT_PATH is not a directory:\n"
        f"{LOG_AUDIT_PATH}"
    )


# ------------------------------------------
# Google Drive-tolerant JSON writer
# ------------------------------------------

def save_registry_json(
    file_path: Path,
    data: dict,
) -> None:
    """
    Write registry JSON using a temporary file first.

    If Google Drive does not support the replacement operation
    reliably, fall back to a direct UTF-8 write.
    """
    if not file_path.parent.exists():
        file_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

    temporary_path = file_path.with_name(
        f".{file_path.name}.tmp"
    )

    try:
        with temporary_path.open(
            mode="w",
            encoding="utf-8",
        ) as temporary_file:
            json.dump(
                data,
                temporary_file,
                indent=4,
                ensure_ascii=False,
            )
            temporary_file.write("\n")

        try:
            temporary_path.replace(file_path)

        except OSError as replace_error:
            log_event(
                message=(
                    "REGISTRY_ATOMIC_REPLACE_FALLBACK: "
                    f"path={file_path}; "
                    f"error={replace_error}. "
                    "Using direct file write."
                ),
                level="WARNING",
            )

            with file_path.open(
                mode="w",
                encoding="utf-8",
            ) as target_file:
                json.dump(
                    data,
                    target_file,
                    indent=4,
                    ensure_ascii=False,
                )
                target_file.write("\n")

    finally:
        if temporary_path.exists():
            temporary_path.unlink()


# ------------------------------------------
# Define approval workflow
# ------------------------------------------

DEFAULT_APPROVAL_WORKFLOW = {
    "review_required": True,
    "instructions": [
        (
            "Review proposed_schema against the source-data "
            "and database requirements."
        ),
        (
            "Confirm all proposed column names and SQL data "
            "types are correct."
        ),
        (
            "Copy the approved schema from proposed_schema "
            "into active_schema."
        ),
        (
            "Set active_schema_status to APPROVED after "
            "human approval."
        ),
        (
            "Assign an active_schema_version such as 1.0.0."
        ),
        (
            "Update proposed_schema_metadata.status to APPROVED."
        ),
        (
            "Add approved_at and approved_by to "
            "proposed_schema_metadata."
        ),
        (
            "Append an approval record containing the approved "
            "schema to revision_history."
        ),
    ],
    "allowed_active_schema_statuses": [
        "NOT_APPROVED",
        "APPROVED",
    ],
    "approval_note": (
        "The profiler never approves or promotes schemas "
        "automatically. Approval requires human review."
    ),
}


# ------------------------------------------
# Define default registry structures
# ------------------------------------------

DEFAULT_MASTER_REGISTRY = {
    "approval_workflow": deepcopy(
        DEFAULT_APPROVAL_WORKFLOW
    ),
    "active_schema": {},
    "active_schema_version": None,
    "active_schema_status": "NOT_APPROVED",
    "proposed_schema": {},
    "proposed_schema_metadata": None,
    "revision_history": [],
    "run_history": [],
}


DEFAULT_PY_REGISTRY = {
    "schemas_by_file": {},
    "run_history": [],
}


# ------------------------------------------
# Define expected section types
# ------------------------------------------

MASTER_REGISTRY_SECTION_TYPES = {
    "approval_workflow": dict,
    "active_schema": dict,
    "active_schema_version": (str, type(None)),
    "active_schema_status": str,
    "proposed_schema": dict,
    "proposed_schema_metadata": (dict, type(None)),
    "revision_history": list,
    "run_history": list,
}


APPROVAL_WORKFLOW_FIELD_TYPES = {
    "review_required": bool,
    "instructions": list,
    "allowed_active_schema_statuses": list,
    "approval_note": str,
}


PY_REGISTRY_SECTION_TYPES = {
    "schemas_by_file": dict,
    "run_history": list,
}


ALLOWED_ACTIVE_SCHEMA_STATUSES = {
    "NOT_APPROVED",
    "APPROVED",
}


# ------------------------------------------
# Create registry when missing
# ------------------------------------------

def create_registry_if_missing(
    file_path: Path,
    default_structure: dict,
    registry_name: str,
) -> bool:
    """
    Create a registry using its complete default structure only
    when the configured file does not already exist.
    """
    if file_path.exists():

        if not file_path.is_file():
            raise FileExistsError(
                f"{registry_name} path exists but is not a file:\n"
                f"{file_path}"
            )

        log_event(
            message=(
                f"{registry_name}_FOUND: "
                f"Existing file preserved. path={file_path}"
            ),
            level="INFO",
        )

        return False

    save_registry_json(
        file_path=file_path,
        data=deepcopy(default_structure),
    )

    if not file_path.exists():
        raise OSError(
            f"{registry_name} creation did not produce a file:\n"
            f"{file_path}"
        )

    log_event(
        message=(
            f"{registry_name}_CREATED: "
            f"path={file_path}."
        ),
        level="INFO",
    )

    return True


# ------------------------------------------
# Load registry conservatively
# ------------------------------------------

def load_registry(
    file_path: Path,
    default_structure: dict,
    registry_name: str,
) -> tuple[dict, bool]:
    """
    Load a registry while preserving malformed source files.

    Returns:
        registry_data
        loaded_from_disk
    """
    try:
        with file_path.open(
            mode="r",
            encoding="utf-8",
        ) as registry_file:
            registry_data = json.load(registry_file)

    except json.JSONDecodeError as error:
        log_event(
            message=(
                f"{registry_name}_INVALID_JSON: "
                f"path={file_path}; error={error}. "
                "Existing file preserved; in-memory defaults used."
            ),
            level="CRITICAL",
        )

        return deepcopy(default_structure), False

    except OSError as error:
        log_event(
            message=(
                f"{registry_name}_READ_FAILED: "
                f"path={file_path}; error={error}. "
                "In-memory defaults used."
            ),
            level="CRITICAL",
        )

        return deepcopy(default_structure), False

    if not isinstance(registry_data, dict):
        log_event(
            message=(
                f"{registry_name}_INVALID_STRUCTURE: "
                "Registry must contain a JSON object. "
                "Existing file preserved; in-memory defaults used."
            ),
            level="CRITICAL",
        )

        return deepcopy(default_structure), False

    return registry_data, True


# ------------------------------------------
# Validate top-level sections
# ------------------------------------------

def validate_registry_structure(
    registry_data: dict,
    default_structure: dict,
    required_section_types: dict[str, Any],
    registry_name: str,
) -> tuple[dict, list[str], list[str]]:
    """
    Add missing sections safely.

    Invalid section types are replaced in memory only and are not
    written over the existing file automatically.
    """
    validated_registry = deepcopy(registry_data)

    missing_sections = []
    invalid_sections = []

    for section_name, expected_type in (
        required_section_types.items()
    ):
        if section_name not in validated_registry:
            validated_registry[section_name] = deepcopy(
                default_structure[section_name]
            )

            missing_sections.append(section_name)
            continue

        if not isinstance(
            validated_registry[section_name],
            expected_type,
        ):
            validated_registry[section_name] = deepcopy(
                default_structure[section_name]
            )

            invalid_sections.append(section_name)

    if missing_sections:
        log_event(
            message=(
                f"{registry_name}_MISSING_SECTIONS: "
                + ", ".join(sorted(missing_sections))
            ),
            level="WARNING",
        )

    if invalid_sections:
        log_event(
            message=(
                f"{registry_name}_INVALID_SECTIONS: "
                + ", ".join(sorted(invalid_sections))
                + ". Defaults applied in memory only."
            ),
            level="CRITICAL",
        )

    return (
        validated_registry,
        missing_sections,
        invalid_sections,
    )


# ------------------------------------------
# Migrate approval workflow
# ------------------------------------------

def migrate_approval_workflow(
    master_registry_data: dict,
) -> tuple[list[str], list[str]]:
    """
    Add missing fields inside approval_workflow.

    Invalid field types are replaced in memory only.
    """
    approval_workflow = master_registry_data[
        "approval_workflow"
    ]

    missing_fields = []
    invalid_fields = []

    for field_name, expected_type in (
        APPROVAL_WORKFLOW_FIELD_TYPES.items()
    ):
        if field_name not in approval_workflow:
            approval_workflow[field_name] = deepcopy(
                DEFAULT_APPROVAL_WORKFLOW[field_name]
            )

            missing_fields.append(field_name)
            continue

        if not isinstance(
            approval_workflow[field_name],
            expected_type,
        ):
            approval_workflow[field_name] = deepcopy(
                DEFAULT_APPROVAL_WORKFLOW[field_name]
            )

            invalid_fields.append(field_name)

    if missing_fields:
        log_event(
            message=(
                "MASTER_REGISTRY_APPROVAL_WORKFLOW_MIGRATED: "
                + ", ".join(sorted(missing_fields))
            ),
            level="WARNING",
        )

    if invalid_fields:
        log_event(
            message=(
                "MASTER_REGISTRY_APPROVAL_WORKFLOW_INVALID: "
                + ", ".join(sorted(invalid_fields))
                + ". Defaults applied in memory only."
            ),
            level="CRITICAL",
        )

    return missing_fields, invalid_fields


# ------------------------------------------
# Validate approval instructions
# ------------------------------------------

def validate_approval_instructions(
    master_registry_data: dict,
) -> list[str]:
    """
    Ensure approval instructions and allowed status values contain
    usable strings.
    """
    invalid_instruction_entries = []

    approval_workflow = master_registry_data[
        "approval_workflow"
    ]

    instructions = approval_workflow[
        "instructions"
    ]

    for index, instruction in enumerate(instructions):
        if not isinstance(instruction, str) or not instruction.strip():
            invalid_instruction_entries.append(
                f"instructions[{index}]"
            )

    allowed_statuses = approval_workflow[
        "allowed_active_schema_statuses"
    ]

    for index, status in enumerate(allowed_statuses):
        if not isinstance(status, str) or not status.strip():
            invalid_instruction_entries.append(
                f"allowed_active_schema_statuses[{index}]"
            )

    if invalid_instruction_entries:
        log_event(
            message=(
                "MASTER_REGISTRY_APPROVAL_CONTENT_INVALID: "
                + ", ".join(invalid_instruction_entries)
            ),
            level="CRITICAL",
        )

    return invalid_instruction_entries


# ------------------------------------------
# Validate master registry contract
# ------------------------------------------

def validate_master_registry_status(
    master_registry_data: dict,
) -> None:
    """
    Validate relationships between status, active schema,
    proposed schema, and schema version.
    """
    active_status = master_registry_data[
        "active_schema_status"
    ]

    active_schema = master_registry_data[
        "active_schema"
    ]

    active_version = master_registry_data[
        "active_schema_version"
    ]

    proposed_schema = master_registry_data[
        "proposed_schema"
    ]

    proposed_metadata = master_registry_data[
        "proposed_schema_metadata"
    ]

    if active_status not in ALLOWED_ACTIVE_SCHEMA_STATUSES:
        log_event(
            message=(
                "MASTER_REGISTRY_UNKNOWN_STATUS: "
                f"active_schema_status={active_status}. "
                "Using NOT_APPROVED in memory."
            ),
            level="WARNING",
        )

        master_registry_data[
            "active_schema_status"
        ] = "NOT_APPROVED"

    if (
        master_registry_data["active_schema_status"]
        == "APPROVED"
        and not active_schema
    ):
        log_event(
            message=(
                "MASTER_REGISTRY_STATUS_CONFLICT: "
                "Status is APPROVED but active_schema is empty. "
                "Using NOT_APPROVED in memory."
            ),
            level="CRITICAL",
        )

        master_registry_data[
            "active_schema_status"
        ] = "NOT_APPROVED"

    if active_schema and active_version is None:
        log_event(
            message=(
                "MASTER_REGISTRY_VERSION_MISSING: "
                "active_schema exists without "
                "active_schema_version."
            ),
            level="WARNING",
        )

    if not active_schema and active_version is not None:
        log_event(
            message=(
                "MASTER_REGISTRY_VERSION_CONFLICT: "
                "active_schema_version exists while "
                "active_schema is empty."
            ),
            level="WARNING",
        )

    if proposed_schema and proposed_metadata is None:
        log_event(
            message=(
                "MASTER_REGISTRY_PROPOSAL_METADATA_MISSING: "
                "proposed_schema exists without metadata."
            ),
            level="WARNING",
        )

    if (
        isinstance(proposed_metadata, dict)
        and proposed_metadata.get("status") == "APPROVED"
        and master_registry_data["active_schema_status"]
        != "APPROVED"
    ):
        log_event(
            message=(
                "MASTER_REGISTRY_APPROVAL_CONFLICT: "
                "Proposal metadata is APPROVED while active "
                "schema status is not APPROVED."
            ),
            level="WARNING",
        )


# ------------------------------------------
# Validate observed Pandas registry records
# ------------------------------------------

def validate_py_schema_entries(
    py_registry_data: dict,
) -> None:
    """
    Report malformed file records without deleting them.
    """
    schemas_by_file = py_registry_data[
        "schemas_by_file"
    ]

    for file_name, file_record in schemas_by_file.items():

        if not isinstance(file_record, dict):
            log_event(
                message=(
                    "PY_REGISTRY_INVALID_FILE_RECORD: "
                    f"file={file_name}; expected JSON object."
                ),
                level="WARNING",
            )

            continue

        latest_observation = file_record.get(
            "latest_observation"
        )

        observation_history = file_record.get(
            "observation_history"
        )

        if (
            latest_observation is not None
            and not isinstance(latest_observation, dict)
        ):
            log_event(
                message=(
                    "PY_REGISTRY_INVALID_LATEST_OBSERVATION: "
                    f"file={file_name}."
                ),
                level="WARNING",
            )

        if (
            observation_history is not None
            and not isinstance(observation_history, list)
        ):
            log_event(
                message=(
                    "PY_REGISTRY_INVALID_OBSERVATION_HISTORY: "
                    f"file={file_name}."
                ),
                level="WARNING",
            )


# ------------------------------------------
# Display resolved registry paths
# ------------------------------------------

print("========== Registry Path Check ==========\n")
print(f"LOG_AUDIT_PATH     : {LOG_AUDIT_PATH}")
print(f"MASTER_SCHEMA_PATH : {MASTER_SCHEMA_PATH}")
print(f"PY_SCHEMA_PATH     : {PY_SCHEMA_PATH}")
print(
    f"Master file exists : "
    f"{MASTER_SCHEMA_PATH.exists()}"
)
print(
    f"Pandas file exists : "
    f"{PY_SCHEMA_PATH.exists()}\n"
)


# ------------------------------------------
# Create missing registry files
# ------------------------------------------

master_registry_created = create_registry_if_missing(
    file_path=MASTER_SCHEMA_PATH,
    default_structure=DEFAULT_MASTER_REGISTRY,
    registry_name="MASTER_REGISTRY",
)

py_registry_created = create_registry_if_missing(
    file_path=PY_SCHEMA_PATH,
    default_structure=DEFAULT_PY_REGISTRY,
    registry_name="PY_REGISTRY",
)


# ------------------------------------------
# Load registry files
# ------------------------------------------

master_registry_raw, master_loaded_from_disk = load_registry(
    file_path=MASTER_SCHEMA_PATH,
    default_structure=DEFAULT_MASTER_REGISTRY,
    registry_name="MASTER_REGISTRY",
)

py_registry_raw, py_loaded_from_disk = load_registry(
    file_path=PY_SCHEMA_PATH,
    default_structure=DEFAULT_PY_REGISTRY,
    registry_name="PY_REGISTRY",
)


# ------------------------------------------
# Validate top-level structures
# ------------------------------------------

(
    master_registry,
    master_missing_sections,
    master_invalid_sections,
) = validate_registry_structure(
    registry_data=master_registry_raw,
    default_structure=DEFAULT_MASTER_REGISTRY,
    required_section_types=MASTER_REGISTRY_SECTION_TYPES,
    registry_name="MASTER_REGISTRY",
)


(
    py_registry,
    py_missing_sections,
    py_invalid_sections,
) = validate_registry_structure(
    registry_data=py_registry_raw,
    default_structure=DEFAULT_PY_REGISTRY,
    required_section_types=PY_REGISTRY_SECTION_TYPES,
    registry_name="PY_REGISTRY",
)


# ------------------------------------------
# Migrate approval workflow
# ------------------------------------------

approval_missing_fields = []
approval_invalid_fields = []

if "approval_workflow" not in master_invalid_sections:
    (
        approval_missing_fields,
        approval_invalid_fields,
    ) = migrate_approval_workflow(
        master_registry_data=master_registry,
    )


approval_invalid_content = []

if not approval_invalid_fields:
    approval_invalid_content = (
        validate_approval_instructions(
            master_registry_data=master_registry,
        )
    )


# ------------------------------------------
# Persist safe structural migrations
# ------------------------------------------

master_migration_required = (
    master_loaded_from_disk
    and (
        bool(master_missing_sections)
        or bool(approval_missing_fields)
    )
    and not master_invalid_sections
    and not approval_invalid_fields
    and not approval_invalid_content
)


if master_migration_required:
    save_registry_json(
        file_path=MASTER_SCHEMA_PATH,
        data=master_registry,
    )

    log_event(
        message=(
            "MASTER_REGISTRY_MIGRATION_SAVED: "
            f"path={MASTER_SCHEMA_PATH}."
        ),
        level="WARNING",
    )


py_migration_required = (
    py_loaded_from_disk
    and bool(py_missing_sections)
    and not py_invalid_sections
)


if py_migration_required:
    save_registry_json(
        file_path=PY_SCHEMA_PATH,
        data=py_registry,
    )

    log_event(
        message=(
            "PY_REGISTRY_MIGRATION_SAVED: "
            f"path={PY_SCHEMA_PATH}."
        ),
        level="WARNING",
    )


# ------------------------------------------
# Validate registry-specific contracts
# ------------------------------------------

validate_master_registry_status(
    master_registry_data=master_registry,
)

validate_py_schema_entries(
    py_registry_data=py_registry,
)


# ------------------------------------------
# Confirm files exist after processing
# ------------------------------------------

if not MASTER_SCHEMA_PATH.exists():
    raise FileNotFoundError(
        "master_schema.json was not created:\n"
        f"{MASTER_SCHEMA_PATH}"
    )

if not PY_SCHEMA_PATH.exists():
    raise FileNotFoundError(
        "py_schema.json was not created:\n"
        f"{PY_SCHEMA_PATH}"
    )


# ------------------------------------------
# Determine registry readiness
# ------------------------------------------

master_registry_ready = (
    not master_invalid_sections
    and not approval_invalid_fields
    and not approval_invalid_content
)

py_registry_ready = not py_invalid_sections


if master_registry_ready:
    log_event(
        message=(
            "MASTER_REGISTRY_READY: "
            "Master schema registry is available."
        ),
        level="INFO",
    )


if py_registry_ready:
    log_event(
        message=(
            "PY_REGISTRY_READY: "
            "Observed Pandas schema registry is available."
        ),
        level="INFO",
    )


# ------------------------------------------
# Display registry summary
# ------------------------------------------

print("\n========== Schema Registry Initialisation ==========\n")

print(f"Master registry path       : {MASTER_SCHEMA_PATH}")
print(
    "Master registry created    : "
    f"{master_registry_created}"
)
print(
    "Approval review required   : "
    f"{master_registry['approval_workflow']['review_required']}"
)
print(
    "Approval instruction count : "
    f"{len(master_registry['approval_workflow']['instructions'])}"
)
print(
    "Active schema status       : "
    f"{master_registry['active_schema_status']}"
)
print(
    "Active schema version      : "
    f"{master_registry['active_schema_version']}"
)
print(
    "Active schema columns      : "
    f"{len(master_registry['active_schema'])}"
)
print(
    "Proposed schema columns    : "
    f"{len(master_registry['proposed_schema'])}"
)
print(
    "Revision records           : "
    f"{len(master_registry['revision_history'])}"
)
print(
    "Master run records         : "
    f"{len(master_registry['run_history'])}"
)

print()

print(f"Pandas registry path       : {PY_SCHEMA_PATH}")
print(
    "Pandas registry created    : "
    f"{py_registry_created}"
)
print(
    "Files with observations    : "
    f"{len(py_registry['schemas_by_file'])}"
)
print(
    "Pandas run records         : "
    f"{len(py_registry['run_history'])}"
)

print("\n" + "=" * 70)

if master_registry_ready:
    print("✓ Master schema registry created, loaded, or migrated.")
else:
    print("⚠ Master schema registry requires human review.")

if py_registry_ready:
    print("✓ Pandas schema registry created, loaded, or migrated.")
else:
    print("⚠ Pandas schema registry requires human review.")

print("✓ Registry files exist at the resolved Google Drive paths.")
print("✓ Human approval instructions are available.")
print("=" * 70)

## Part 2.4: Discover and Validate Source CSV Files

### Objective

Discover all CSV files available in the configured raw-data directory and confirm that the profiler has valid source files to inspect.

The source directory was resolved earlier as `RAW_DATA_PATH`. This section reuses that path and does not reconstruct project configuration or Google Drive paths.

### File-Discovery Policy

The profiler searches only for files with the `.csv` extension located directly inside `RAW_DATA_PATH`.

The discovered files are sorted by filename before processing. Deterministic sorting is necessary because the first successfully profiled file may later be used to generate the proposed SQL schema when no approved active schema exists.

The profiler does not rely on the unspecified order returned by the filesystem.

### Prerequisite Rules

Before profiling begins:

* `RAW_DATA_PATH` must exist;
* `RAW_DATA_PATH` must be a directory;
* at least one CSV file must be available.

If the directory is missing or no CSV files are found, the profiler records a `CRITICAL` audit event and stops before schema profiling begins.

An empty source directory must not produce a misleading successful profiler result.

### Process

This step will:

1. Verify that `RAW_DATA_PATH` exists.
2. Verify that `RAW_DATA_PATH` is a directory.
3. Discover files matching `*.csv`.
4. Sort the discovered files by filename.
5. Store the sorted file paths in `discovered_files`.
6. Update `files_discovered_count`.
7. Record the discovery result in `pipeline_audit.log`.
8. Display the ordered file list in the notebook.

This section does not open or validate the contents of the CSV files. File-level parsing and schema inference will occur in Part 2.5.

### Expected Outcome

* One or more CSV files are discovered.
* The file list is sorted deterministically.
* `discovered_files` contains reusable `Path` objects.
* `files_discovered_count` reflects the number of source files found.
* A `FILE_DISCOVERY_COMPLETE` event is appended to the audit log.
* The profiler is ready to inspect each source file in Part 2.5.


In [ ]:
# ==========================================
# Part 2.4: Discover and Validate
# Source CSV Files
# ==========================================

from pathlib import Path


# ------------------------------------------
# Discover source CSV files
# ------------------------------------------

def discover_csv_files(
    source_directory: Path,
) -> list[Path]:
    """
    Discover CSV files directly inside the configured source
    directory and return them in deterministic filename order.
    """
    if not source_directory.exists():
        log_event(
            message=(
                "FILE_DISCOVERY_FAILED: RAW_DATA_PATH does not "
                f"exist. Path={source_directory}"
            ),
            level="CRITICAL",
        )

        raise FileNotFoundError(
            f"RAW_DATA_PATH does not exist:\n{source_directory}"
        )

    if not source_directory.is_dir():
        log_event(
            message=(
                "FILE_DISCOVERY_FAILED: RAW_DATA_PATH is not "
                f"a directory. Path={source_directory}"
            ),
            level="CRITICAL",
        )

        raise NotADirectoryError(
            f"RAW_DATA_PATH is not a directory:\n"
            f"{source_directory}"
        )

    csv_files = sorted(
        (
            file_path
            for file_path in source_directory.glob("*.csv")
            if file_path.is_file()
        ),
        key=lambda file_path: file_path.name.lower(),
    )

    if not csv_files:
        log_event(
            message=(
                "FILE_DISCOVERY_FAILED: No CSV files were found "
                f"in RAW_DATA_PATH. Path={source_directory}"
            ),
            level="CRITICAL",
        )

        raise FileNotFoundError(
            "No CSV files were found for profiling in:\n"
            f"{source_directory}"
        )

    return csv_files


# ------------------------------------------
# Run source-file discovery
# ------------------------------------------

log_event(
    message=(
        "FILE_DISCOVERY_START: Searching for CSV files in "
        f"RAW_DATA_PATH={RAW_DATA_PATH}"
    ),
    level="INFO",
)

discovered_files = discover_csv_files(
    source_directory=RAW_DATA_PATH,
)

files_discovered_count = len(discovered_files)


# ------------------------------------------
# Record successful discovery
# ------------------------------------------

log_event(
    message=(
        "FILE_DISCOVERY_COMPLETE: "
        f"files_discovered={files_discovered_count}; "
        "processing_order=sorted_by_filename."
    ),
    level="INFO",
)


# ------------------------------------------
# Display discovered source files
# ------------------------------------------

print("\n========== Source CSV Discovery ==========\n")

print(f"Source directory : {RAW_DATA_PATH}")
print(f"Files discovered : {files_discovered_count}")
print("Processing order : Sorted by filename\n")

for file_number, file_path in enumerate(
    discovered_files,
    start=1,
):
    print(f"{file_number:>3}. {file_path.name}")


print("\n" + "=" * 55)
print("✓ Source CSV files discovered.")
print("✓ File processing order is deterministic.")
print("✓ Files are ready for schema profiling.")
print("=" * 55)

## Part 2.5: Profile CSV Files and Detect Schema Differences

### Objective

Inspect every discovered CSV file, record its observed Pandas schema, translate that schema into SQL-compatible types, and compare it against the applicable schema baseline.

This is the core diagnostic stage of the profiler.

### Profiling Policy

Each file is processed independently so that one unreadable file does not prevent the remaining files from being inspected.

The profiler uses the sample policy configured through `PROFILE_SAMPLE_ROWS`:

* a positive integer reads that number of rows;
* `0` or `null` reads the complete file.

Sample-based results are labelled as sampled observations and must not be treated as guaranteed full-file validation.

### CSV Read Strategy

Before reading the data sample, the profiler reads the CSV header to identify the columns that actually exist.

This allows the profiler to:

* apply explicit string dtypes only to available columns;
* parse known datetime columns only when present;
* avoid failing solely because a known datetime column is missing;
* classify missing columns as structural differences rather than immediate parsing failures.

Known datetime columns currently include:

```text
started_at
ended_at
```

### Schema Translation

For each successfully read file, the profiler records:

* the observed Pandas dtype for every column;
* the translated SQL-compatible type;
* any exact column-specific SQL overrides;
* the profiling mode and sample size;
* the number of rows inspected.

Unknown Pandas dtypes generate a warning. A configured object-type fallback is used only when available.

### Baseline Policy

If an approved active schema exists, each file is compared against that schema and differences are classified as formal schema drift.

If no approved active schema exists:

* an existing proposed schema is used as the diagnostic comparison baseline;
* otherwise, the first successfully profiled file creates an in-memory proposed schema;
* the proposal remains `PENDING_REVIEW`;
* the active schema remains unchanged;
* differences against the proposal are classified as proposed-schema variance rather than approved-schema drift.

### Difference Categories

Schema comparison identifies:

* `TYPE_MISMATCH` — the column exists but its SQL type differs;
* `MISSING_COLUMN` — the baseline expects a column that is absent;
* `NEW_COLUMN` — the source contains a column not present in the baseline.

Schema differences and file-processing failures are tracked separately.

### Persistence Boundary

This section updates `master_registry`, observed schema collections, counters, and diagnostic details in memory.

It does not write the final registry changes to disk. Atomic persistence, run-history updates, and the final profiler summary will be handled in Part 2.6.

### Expected Outcome

* Every discovered CSV file is attempted.
* Successfully read files have Pandas and SQL schemas recorded.
* Unreadable files are recorded separately as processing failures.
* A proposed schema is generated when no approved or proposed schema exists.
* Formal drift and proposed-schema variance are classified separately.
* In-memory results are ready for persistence in Part 2.6.


In [ ]:
# ==========================================
# Part 2.5: Profile CSV Files and Detect
# Schema Differences
# ==========================================

from datetime import datetime
from pathlib import Path

import pandas as pd


# ------------------------------------------
# Profiling configuration
# ------------------------------------------

KNOWN_DATETIME_COLUMNS = {
    "started_at",
    "ended_at",
}


# ------------------------------------------
# Read CSV header
# ------------------------------------------

def read_csv_columns(
    file_path: Path,
) -> list[str]:
    """
    Read only the CSV header so dtype and datetime rules can be
    limited to columns that are actually present.
    """
    header_frame = pd.read_csv(
        file_path,
        nrows=0,
    )

    return list(header_frame.columns)


# ------------------------------------------
# Build file-specific read rules
# ------------------------------------------

def build_csv_read_options(
    available_columns: list[str],
) -> tuple[dict, list[str]]:
    """
    Limit configured dtype and datetime rules to columns that
    exist in the current file.
    """
    available_column_set = set(available_columns)

    file_explicit_dtypes = {
        column_name: dtype_value
        for column_name, dtype_value in EXPLICIT_READ_DTYPES.items()
        if column_name in available_column_set
    }

    file_datetime_columns = sorted(
        KNOWN_DATETIME_COLUMNS & available_column_set
    )

    return file_explicit_dtypes, file_datetime_columns


# ------------------------------------------
# Read CSV sample or complete file
# ------------------------------------------

def read_csv_for_profiling(
    file_path: Path,
    explicit_dtypes: dict,
    datetime_columns: list[str],
) -> pd.DataFrame:
    """
    Read the configured sample or complete CSV file.
    """
    read_options = {
        "filepath_or_buffer": file_path,
        "dtype": explicit_dtypes or None,
        "parse_dates": datetime_columns or None,
        "low_memory": False,
    }

    if EFFECTIVE_SAMPLE_ROWS is not None:
        read_options["nrows"] = EFFECTIVE_SAMPLE_ROWS

    return pd.read_csv(**read_options)


# ------------------------------------------
# Record observed Pandas schema
# ------------------------------------------

def extract_pandas_schema(
    dataframe: pd.DataFrame,
) -> dict[str, str]:
    """
    Return the observed Pandas dtype for every DataFrame column.
    """
    return {
        column_name: str(dataframe[column_name].dtype)
        for column_name in dataframe.columns
    }


# ------------------------------------------
# Translate Pandas schema to SQL types
# ------------------------------------------

def map_pandas_to_sql_types(
    pandas_schema: dict[str, str],
    file_name: str,
) -> dict[str, str]:
    """
    Translate observed Pandas dtypes using the validated type map.

    Unknown dtypes are logged before the configured object fallback
    is applied.
    """
    sql_schema = {}

    fallback_sql_type = PANDAS_TO_SQL_TYPE_MAP.get(
        "object"
    )

    for column_name, pandas_dtype in pandas_schema.items():

        if pandas_dtype in PANDAS_TO_SQL_TYPE_MAP:
            sql_schema[column_name] = (
                PANDAS_TO_SQL_TYPE_MAP[pandas_dtype]
            )

            continue

        if fallback_sql_type is None:
            raise KeyError(
                "No SQL mapping exists for Pandas dtype "
                f"'{pandas_dtype}', and no object fallback is "
                "configured."
            )

        log_event(
            message=(
                "UNMAPPED_PANDAS_DTYPE: "
                f"file={file_name}; "
                f"column={column_name}; "
                f"pandas_dtype={pandas_dtype}; "
                f"fallback_sql_type={fallback_sql_type}."
            ),
            level="WARNING",
        )

        sql_schema[column_name] = fallback_sql_type

    return sql_schema


# ------------------------------------------
# Apply exact SQL type overrides
# ------------------------------------------

def apply_sql_type_overrides(
    sql_schema: dict[str, str],
) -> dict[str, str]:
    """
    Apply column-specific SQL overrides using exact column-name
    matching only.
    """
    overridden_schema = dict(sql_schema)

    for column_name, override_type in (
        COLUMN_SQL_TYPE_OVERRIDES.items()
    ):
        if column_name in overridden_schema:
            overridden_schema[column_name] = override_type

    return overridden_schema


# ------------------------------------------
# Compare observed and baseline schemas
# ------------------------------------------

def compare_schemas(
    baseline_schema: dict[str, str],
    observed_schema: dict[str, str],
) -> dict[str, dict]:
    """
    Compare two SQL schemas and return detailed structural
    differences.
    """
    differences = {}

    for column_name, expected_type in baseline_schema.items():

        if column_name not in observed_schema:
            differences[column_name] = {
                "issue": "MISSING_COLUMN",
                "expected": expected_type,
                "observed": None,
            }

        elif observed_schema[column_name] != expected_type:
            differences[column_name] = {
                "issue": "TYPE_MISMATCH",
                "expected": expected_type,
                "observed": observed_schema[column_name],
            }

    for column_name, observed_type in observed_schema.items():

        if column_name not in baseline_schema:
            differences[column_name] = {
                "issue": "NEW_COLUMN",
                "expected": None,
                "observed": observed_type,
            }

    return differences


# ------------------------------------------
# Determine available comparison baseline
# ------------------------------------------

approved_schema_available = (
    master_registry["active_schema_status"] == "APPROVED"
    and bool(master_registry["active_schema"])
)

if approved_schema_available:
    schema_comparison_mode = "APPROVED_ACTIVE_SCHEMA"
    comparison_schema = master_registry["active_schema"]

elif master_registry["proposed_schema"]:
    schema_comparison_mode = "EXISTING_PROPOSED_SCHEMA"
    comparison_schema = master_registry["proposed_schema"]

else:
    schema_comparison_mode = "DISCOVERY"
    comparison_schema = None


proposal_created_this_run = False
proposal_source_file = None


log_event(
    message=(
        "FILE_PROFILING_START: "
        f"files_to_profile={files_discovered_count}; "
        f"comparison_mode={schema_comparison_mode}."
    ),
    level="INFO",
)


# ------------------------------------------
# Profile every discovered CSV file
# ------------------------------------------

for file_path in discovered_files:

    file_name = file_path.name

    log_event(
        message=f"FILE_START: {file_name}",
        level="INFO",
    )

    try:
        available_columns = read_csv_columns(
            file_path=file_path,
        )

        if not available_columns:
            raise ValueError(
                "CSV file contains no column headers."
            )

        (
            file_explicit_dtypes,
            file_datetime_columns,
        ) = build_csv_read_options(
            available_columns=available_columns,
        )

        dataframe = read_csv_for_profiling(
            file_path=file_path,
            explicit_dtypes=file_explicit_dtypes,
            datetime_columns=file_datetime_columns,
        )

        pandas_schema = extract_pandas_schema(
            dataframe=dataframe,
        )

        translated_sql_schema = map_pandas_to_sql_types(
            pandas_schema=pandas_schema,
            file_name=file_name,
        )

        sql_schema = apply_sql_type_overrides(
            sql_schema=translated_sql_schema,
        )

        observation_time = datetime.now(
            MELBOURNE_TIMEZONE
        ).isoformat(timespec="seconds")

        file_observation = {
            "session_id": SESSION_ID,
            "observed_at": observation_time,
            "profiling_mode": PROFILING_MODE,
            "sample_rows": EFFECTIVE_SAMPLE_ROWS,
            "rows_profiled": len(dataframe),
            "columns_profiled": len(dataframe.columns),
            "pandas_schema": pandas_schema,
        }

        observed_pandas_schemas_by_file[
            file_name
        ] = file_observation

        observed_sql_schemas_by_file[
            file_name
        ] = sql_schema

        processed_files.append(file_name)
        files_processed_successfully_count += 1


        # ----------------------------------
        # Create proposed schema if required
        # ----------------------------------

        if comparison_schema is None:
            master_registry["proposed_schema"] = dict(
                sql_schema
            )

            master_registry[
                "proposed_schema_metadata"
            ] = {
                "status": "PENDING_REVIEW",
                "source_file": file_name,
                "session_id": SESSION_ID,
                "created_at": observation_time,
            }

            comparison_schema = master_registry[
                "proposed_schema"
            ]

            schema_comparison_mode = (
                "NEW_PROPOSED_SCHEMA"
            )

            proposal_created_this_run = True
            proposal_source_file = file_name

            log_event(
                message=(
                    "PROPOSED_SCHEMA_CREATED: "
                    f"source_file={file_name}; "
                    f"columns={len(sql_schema)}; "
                    "status=PENDING_REVIEW."
                ),
                level="INFO",
            )

            # The source file defines the proposal and therefore
            # is not compared against itself.
            file_differences = {}


        # ----------------------------------
        # Compare against applicable schema
        # ----------------------------------

        else:
            file_differences = compare_schemas(
                baseline_schema=comparison_schema,
                observed_schema=sql_schema,
            )


        # ----------------------------------
        # Classify schema differences
        # ----------------------------------

        if file_differences:

            if approved_schema_available:
                drift_details_by_file[
                    file_name
                ] = file_differences

                files_with_drift_count += 1

                log_event(
                    message=(
                        "SCHEMA_DRIFT_DETECTED: "
                        f"file={file_name}; "
                        f"difference_count="
                        f"{len(file_differences)}."
                    ),
                    level="WARNING",
                )

            else:
                proposed_schema_variance_by_file[
                    file_name
                ] = file_differences

                files_with_proposed_schema_variance_count += 1

                log_event(
                    message=(
                        "PROPOSED_SCHEMA_VARIANCE: "
                        f"file={file_name}; "
                        f"difference_count="
                        f"{len(file_differences)}."
                    ),
                    level="WARNING",
                )

        else:
            comparison_label = (
                "approved active schema"
                if approved_schema_available
                else "proposed schema"
            )

            log_event(
                message=(
                    "SCHEMA_MATCH: "
                    f"file={file_name}; "
                    f"baseline={comparison_label}."
                ),
                level="INFO",
            )


        log_event(
            message=(
                "FILE_COMPLETE: "
                f"file={file_name}; "
                f"rows_profiled={len(dataframe)}; "
                f"columns_profiled={len(dataframe.columns)}."
            ),
            level="INFO",
        )


    except (
        pd.errors.EmptyDataError,
        pd.errors.ParserError,
        UnicodeDecodeError,
        OSError,
        ValueError,
        KeyError,
        TypeError,
    ) as error:

        failure_time = datetime.now(
            MELBOURNE_TIMEZONE
        ).isoformat(timespec="seconds")

        failed_files.append(
            {
                "file_name": file_name,
                "failed_at": failure_time,
                "error_type": type(error).__name__,
                "error_message": str(error),
            }
        )

        files_failed_count += 1

        log_event(
            message=(
                "FILE_FAILED: "
                f"file={file_name}; "
                f"error_type={type(error).__name__}; "
                f"error={error}"
            ),
            level="ERROR",
        )


# ------------------------------------------
# Record profiling-stage completion
# ------------------------------------------

log_event(
    message=(
        "FILE_PROFILING_COMPLETE: "
        f"files_discovered={files_discovered_count}; "
        f"files_processed={files_processed_successfully_count}; "
        f"files_failed={files_failed_count}; "
        f"files_with_drift={files_with_drift_count}; "
        "files_with_proposed_schema_variance="
        f"{files_with_proposed_schema_variance_count}."
    ),
    level="INFO",
)


# ------------------------------------------
# Display profiling summary
# ------------------------------------------

print("\n========== CSV Schema Profiling ==========\n")

print(
    f"Comparison mode          : "
    f"{schema_comparison_mode}"
)
print(
    f"Files discovered         : "
    f"{files_discovered_count}"
)
print(
    f"Files processed          : "
    f"{files_processed_successfully_count}"
)
print(
    f"Files failed             : "
    f"{files_failed_count}"
)
print(
    f"Files with schema drift  : "
    f"{files_with_drift_count}"
)
print(
    "Files with proposal variance: "
    f"{files_with_proposed_schema_variance_count}"
)

if proposal_created_this_run:
    print(
        f"Proposal source file     : "
        f"{proposal_source_file}"
    )

if failed_files:
    print("\nFailed files:")

    for failed_file in failed_files:
        print(
            f"  - {failed_file['file_name']}: "
            f"{failed_file['error_type']} — "
            f"{failed_file['error_message']}"
        )

if drift_details_by_file:
    print("\nApproved-schema drift:")

    for file_name, differences in (
        drift_details_by_file.items()
    ):
        print(
            f"  - {file_name}: "
            f"{len(differences)} difference(s)"
        )

if proposed_schema_variance_by_file:
    print("\nProposed-schema variance:")

    for file_name, differences in (
        proposed_schema_variance_by_file.items()
    ):
        print(
            f"  - {file_name}: "
            f"{len(differences)} difference(s)"
        )


print("\n" + "=" * 65)

if files_processed_successfully_count == 0:
    print("✗ No CSV file was profiled successfully.")
else:
    print("✓ CSV schema profiling completed.")
    print("✓ Pandas and SQL observations are available in memory.")

print("✓ Results are ready for Part 2.6 persistence.")
print("=" * 65)

## Part 2.6: Analyse Schema Differences

### Objective

Analyse the SQL-compatible schemas observed in Part 2.5 and compare each successfully profiled source file against the applicable schema baseline.

This section determines whether the profiler should use:

* an approved active schema;
* an existing proposed schema awaiting review; or
* a new proposed schema generated from the first successfully profiled file.

Schema analysis is performed entirely in memory. Registry persistence and final run reporting will occur in Part 2.7.

### Baseline Selection Policy

The profiler follows this priority order:

1. Use `active_schema` when its status is `APPROVED` and the schema is not empty.
2. Otherwise, use an existing `proposed_schema` when available.
3. Otherwise, use the first successfully profiled file to create a new proposed schema.

The first successful source file is deterministic because source files were sorted by filename in Part 2.4.

### Human-in-the-Loop Policy

A newly detected schema is stored as:

```text
proposed_schema
```

It is not automatically copied into:

```text
active_schema
```

The proposal remains pending until it has been reviewed and approved by a human.

A newly generated proposal receives metadata containing:

* review status;
* source filename;
* profiler session ID;
* Melbourne creation timestamp.

### Difference Classification

For every file, the observed SQL schema is compared column by column with the selected baseline.

The profiler identifies:

* `TYPE_MISMATCH` when a column exists in both schemas but has a different SQL type;
* `MISSING_COLUMN` when the baseline expects a column that is absent from the file;
* `NEW_COLUMN` when the file contains a column that is not present in the baseline.

### Formal Drift and Proposal Variance

The same structural difference has different governance meaning depending on the baseline:

* Differences against an approved active schema are classified as formal schema drift.
* Differences against an unapproved proposed schema are classified as proposed-schema variance.

This distinction prevents unapproved discovery results from being treated as authoritative production contracts.

### Failure Boundary

Only successfully profiled files are analysed.

Files that failed during Part 2.5 remain in `failed_files` and are not classified as schema drift.

### In-Memory Outputs

This section prepares:

```text
drift_details_by_file
proposed_schema_variance_by_file
files_with_drift_count
files_with_proposed_schema_variance_count
schema_comparison_mode
proposal_created_this_run
proposal_source_file
```

It may also update `master_registry["proposed_schema"]` and `master_registry["proposed_schema_metadata"]` in memory.

### Expected Outcome

* The applicable schema baseline is selected explicitly.
* A proposed schema is created when no approved or proposed schema exists.
* Every successfully profiled file is compared against the baseline.
* Formal drift and proposed-schema variance are tracked separately.
* Detailed differences are available by filename.
* Analysis results are ready for persistence and final reporting in Part 2.7.


In [ ]:
# ==========================================
# Part 2.6: Analyse Schema Differences
# ==========================================

from datetime import datetime


# ------------------------------------------
# Compare baseline and observed SQL schemas
# ------------------------------------------

def compare_schemas(
    baseline_schema: dict[str, str],
    observed_schema: dict[str, str],
) -> dict[str, dict]:
    """
    Compare an observed SQL schema with a baseline schema and
    return detailed differences by column.
    """
    differences = {}

    for column_name, expected_type in baseline_schema.items():

        if column_name not in observed_schema:
            differences[column_name] = {
                "issue": "MISSING_COLUMN",
                "expected": expected_type,
                "observed": None,
            }

        elif observed_schema[column_name] != expected_type:
            differences[column_name] = {
                "issue": "TYPE_MISMATCH",
                "expected": expected_type,
                "observed": observed_schema[column_name],
            }

    for column_name, observed_type in observed_schema.items():

        if column_name not in baseline_schema:
            differences[column_name] = {
                "issue": "NEW_COLUMN",
                "expected": None,
                "observed": observed_type,
            }

    return differences


# ------------------------------------------
# Validate analysis prerequisites
# ------------------------------------------

if files_processed_successfully_count == 0:
    log_event(
        message=(
            "SCHEMA_ANALYSIS_FAILED: No successfully profiled "
            "files are available for schema analysis."
        ),
        level="CRITICAL",
    )

    raise RuntimeError(
        "Schema analysis cannot continue because no CSV file "
        "was profiled successfully."
    )


missing_sql_observations = [
    file_name
    for file_name in processed_files
    if file_name not in observed_sql_schemas_by_file
]

if missing_sql_observations:
    log_event(
        message=(
            "SCHEMA_ANALYSIS_FAILED: SQL observations are missing "
            "for successfully processed file(s): "
            + ", ".join(sorted(missing_sql_observations))
        ),
        level="CRITICAL",
    )

    raise RuntimeError(
        "SQL schema observations are incomplete."
    )


# ------------------------------------------
# Reset analysis-stage collections
# ------------------------------------------

drift_details_by_file = {}
proposed_schema_variance_by_file = {}

files_with_drift_count = 0
files_with_proposed_schema_variance_count = 0

proposal_created_this_run = False
proposal_source_file = None


# ------------------------------------------
# Determine applicable schema baseline
# ------------------------------------------

approved_schema_available = (
    master_registry["active_schema_status"] == "APPROVED"
    and bool(master_registry["active_schema"])
)

if approved_schema_available:
    schema_comparison_mode = "APPROVED_ACTIVE_SCHEMA"
    comparison_schema = dict(
        master_registry["active_schema"]
    )

    log_event(
        message=(
            "SCHEMA_BASELINE_SELECTED: "
            "mode=APPROVED_ACTIVE_SCHEMA; "
            f"version={master_registry['active_schema_version']}; "
            f"columns={len(comparison_schema)}."
        ),
        level="INFO",
    )


elif master_registry["proposed_schema"]:
    schema_comparison_mode = "EXISTING_PROPOSED_SCHEMA"
    comparison_schema = dict(
        master_registry["proposed_schema"]
    )

    log_event(
        message=(
            "SCHEMA_BASELINE_SELECTED: "
            "mode=EXISTING_PROPOSED_SCHEMA; "
            f"columns={len(comparison_schema)}."
        ),
        level="INFO",
    )


else:
    schema_comparison_mode = "NEW_PROPOSED_SCHEMA"

    proposal_source_file = processed_files[0]

    proposal_source_observation = (
        observed_sql_schemas_by_file[
            proposal_source_file
        ]
    )

    comparison_schema = dict(
        proposal_source_observation["sql_schema"]
    )

    proposal_created_at = datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds")

    master_registry["proposed_schema"] = dict(
        comparison_schema
    )

    master_registry[
        "proposed_schema_metadata"
    ] = {
        "status": "PENDING_REVIEW",
        "source_file": proposal_source_file,
        "session_id": SESSION_ID,
        "created_at": proposal_created_at,
    }

    proposal_created_this_run = True

    log_event(
        message=(
            "PROPOSED_SCHEMA_CREATED: "
            f"source_file={proposal_source_file}; "
            f"columns={len(comparison_schema)}; "
            "status=PENDING_REVIEW."
        ),
        level="INFO",
    )


# ------------------------------------------
# Record schema-analysis start
# ------------------------------------------

log_event(
    message=(
        "SCHEMA_ANALYSIS_START: "
        f"files_to_compare={files_processed_successfully_count}; "
        f"comparison_mode={schema_comparison_mode}."
    ),
    level="INFO",
)


# ------------------------------------------
# Compare every successfully profiled file
# ------------------------------------------

for file_name in processed_files:

    sql_observation = observed_sql_schemas_by_file[
        file_name
    ]

    observed_sql_schema = sql_observation[
        "sql_schema"
    ]


    # The file that generated a new proposal defines the baseline.
    # Comparing it against itself would add no diagnostic value.
    if (
        proposal_created_this_run
        and file_name == proposal_source_file
    ):
        log_event(
            message=(
                "SCHEMA_BASELINE_SOURCE: "
                f"file={file_name}; "
                "result=PROPOSAL_SOURCE."
            ),
            level="INFO",
        )

        continue


    differences = compare_schemas(
        baseline_schema=comparison_schema,
        observed_schema=observed_sql_schema,
    )


    if not differences:
        baseline_label = (
            "APPROVED_ACTIVE_SCHEMA"
            if approved_schema_available
            else "PROPOSED_SCHEMA"
        )

        log_event(
            message=(
                "SCHEMA_MATCH: "
                f"file={file_name}; "
                f"baseline={baseline_label}."
            ),
            level="INFO",
        )

        continue


    if approved_schema_available:
        drift_details_by_file[file_name] = differences
        files_with_drift_count += 1

        log_event(
            message=(
                "SCHEMA_DRIFT_DETECTED: "
                f"file={file_name}; "
                f"difference_count={len(differences)}."
            ),
            level="WARNING",
        )


    else:
        proposed_schema_variance_by_file[
            file_name
        ] = differences

        files_with_proposed_schema_variance_count += 1

        log_event(
            message=(
                "PROPOSED_SCHEMA_VARIANCE: "
                f"file={file_name}; "
                f"difference_count={len(differences)}."
            ),
            level="WARNING",
        )


# ------------------------------------------
# Record schema-analysis completion
# ------------------------------------------

log_event(
    message=(
        "SCHEMA_ANALYSIS_COMPLETE: "
        f"comparison_mode={schema_comparison_mode}; "
        f"files_with_drift={files_with_drift_count}; "
        "files_with_proposed_schema_variance="
        f"{files_with_proposed_schema_variance_count}."
    ),
    level="INFO",
)


# ------------------------------------------
# Display detailed analysis results
# ------------------------------------------

print("\n========== Schema Difference Analysis ==========\n")

print(
    f"Comparison mode                  : "
    f"{schema_comparison_mode}"
)
print(
    f"Files analysed                   : "
    f"{files_processed_successfully_count}"
)
print(
    f"Files with approved-schema drift : "
    f"{files_with_drift_count}"
)
print(
    "Files with proposal variance     : "
    f"{files_with_proposed_schema_variance_count}"
)

if proposal_created_this_run:
    print(
        f"Proposal source file             : "
        f"{proposal_source_file}"
    )
    print(
        "Proposal status                  : "
        "PENDING_REVIEW"
    )


# ------------------------------------------
# Display formal drift details
# ------------------------------------------

if drift_details_by_file:
    print("\nApproved-schema drift details:")

    for file_name, differences in (
        drift_details_by_file.items()
    ):
        print(f"\n  {file_name}")

        for column_name, detail in differences.items():
            print(
                f"    - {column_name}: "
                f"{detail['issue']}; "
                f"expected={detail['expected']}; "
                f"observed={detail['observed']}"
            )


# ------------------------------------------
# Display proposed-schema variance details
# ------------------------------------------

if proposed_schema_variance_by_file:
    print("\nProposed-schema variance details:")

    for file_name, differences in (
        proposed_schema_variance_by_file.items()
    ):
        print(f"\n  {file_name}")

        for column_name, detail in differences.items():
            print(
                f"    - {column_name}: "
                f"{detail['issue']}; "
                f"expected={detail['expected']}; "
                f"observed={detail['observed']}"
            )


print("\n" + "=" * 70)

if approved_schema_available:

    if files_with_drift_count:
        print("⚠ Approved-schema drift was detected.")
    else:
        print("✓ All successfully profiled files match the approved schema.")

else:

    if proposal_created_this_run:
        print("✓ A proposed schema was generated for human review.")

    if files_with_proposed_schema_variance_count:
        print("⚠ Variance against the proposed schema was detected.")
    else:
        print("✓ All analysed files are consistent with the proposed schema.")

print("✓ Analysis results are ready for Part 2.7 persistence.")
print("=" * 70)

## Part 2.7: Persist Schema Registries and Produce the Final Run Summary

### Objective

Persist the profiler results generated during the current session, update schema-observation histories, determine the final run status, and produce a complete execution summary.

This section is the final stage of the CSV schema diagnostic profiler.

### Registry Updates

The profiler updates two persistent registry files.

#### Master SQL schema registry

`master_schema.json` receives:

* the proposed schema created during discovery, when applicable;
* proposed-schema metadata;
* one run-history record describing the current profiler execution.

The approved active schema and its version are not changed unless a separate human approval process explicitly updates them.

Ordinary profiling runs and detected schema differences must not create new approved schema versions.

#### Observed Pandas schema registry

`py_schema.json` stores observations by filename.

For each successfully profiled file:

1. The existing `latest_observation`, when present, is moved into `observation_history`.
2. The current observation becomes the new `latest_observation`.
3. Existing historical observations are preserved.

The registry also receives one run-history record for the current execution.

### Run Status Policy

The final status is determined from actual execution results.

The status precedence is:

1. `FAILED` — no file was successfully profiled.
2. `PARTIAL_FAILURE` — one or more files failed during profiling.
3. `SCHEMA_DRIFT_DETECTED` — files differed from an approved active schema.
4. `PROPOSED_SCHEMA_VARIANCE_DETECTED` — files differed from an unapproved proposed schema.
5. `SUCCESS` — all successfully processed files were structurally consistent and no processing failures occurred.

`PROPOSED_SCHEMA_VARIANCE_DETECTED` is kept separate from formal schema drift because an unapproved proposal is not yet an authoritative schema contract.

### Persistent Write Policy

Registry files are written using the atomic JSON helper defined earlier:

1. Write the updated content to a temporary file.
2. Complete and close the temporary file.
3. Replace the target registry file.

This reduces the risk of leaving incomplete JSON if execution is interrupted during a write.

Each registry write is atomic individually. The two files do not form a database transaction, so a failure between writes is logged as a critical persistence error.

### Final Summary

The final profiler summary includes:

* session ID;
* Melbourne start and end times;
* duration;
* timezone;
* final status;
* active schema version and status;
* schema comparison mode;
* profiling mode and sample size;
* files discovered;
* files successfully processed;
* files failed;
* files with formal schema drift;
* files with proposed-schema variance;
* warning and error counts;
* detailed drift and variance information.

A final `SESSION_END` event is appended to `pipeline_audit.log`.

### Expected Outcome

* Current file observations are stored in `py_schema.json`.
* Previous latest observations are preserved in history.
* The master and Pandas registries receive run-history records.
* Proposed schema information is persisted without becoming approved automatically.
* Registry files are written safely.
* The final profiler status accurately reflects the execution.
* A complete run summary is displayed.
* The profiler session is formally closed in the audit log.


In [ ]:
# ==========================================
# Part 2.7: Persist Schema Registries and
# Produce Final Run Summary
# ==========================================

from copy import deepcopy
from datetime import datetime


# ------------------------------------------
# Determine final profiler status
# ------------------------------------------

def determine_run_status(
    files_processed: int,
    files_failed: int,
    files_with_drift: int,
    files_with_proposal_variance: int,
) -> str:
    """
    Determine the final status using execution-result precedence.
    """
    if files_processed == 0:
        return "FAILED"

    if files_failed > 0:
        return "PARTIAL_FAILURE"

    if files_with_drift > 0:
        return "SCHEMA_DRIFT_DETECTED"

    if files_with_proposal_variance > 0:
        return "PROPOSED_SCHEMA_VARIANCE_DETECTED"

    return "SUCCESS"


# ------------------------------------------
# Update observed-schema history by file
# ------------------------------------------

def update_py_schema_observations(
    registry: dict,
    current_observations: dict,
) -> None:
    """
    Store the latest Pandas observation for each successfully
    profiled file while preserving its previous latest observation.
    """
    schemas_by_file = registry["schemas_by_file"]

    for file_name, new_observation in current_observations.items():

        existing_file_record = schemas_by_file.get(file_name)

        if not isinstance(existing_file_record, dict):
            existing_file_record = {
                "latest_observation": None,
                "observation_history": [],
            }

        latest_observation = existing_file_record.get(
            "latest_observation"
        )

        observation_history = existing_file_record.get(
            "observation_history",
            [],
        )

        if not isinstance(observation_history, list):
            log_event(
                message=(
                    "PY_REGISTRY_HISTORY_RESET_IN_MEMORY: "
                    f"file={file_name}; observation_history was "
                    "not a list."
                ),
                level="WARNING",
            )

            observation_history = []

        if isinstance(latest_observation, dict):
            observation_history.append(
                deepcopy(latest_observation)
            )

        elif latest_observation is not None:
            log_event(
                message=(
                    "PY_REGISTRY_LATEST_OBSERVATION_SKIPPED: "
                    f"file={file_name}; previous latest observation "
                    "was malformed and was not added to history."
                ),
                level="WARNING",
            )

        schemas_by_file[file_name] = {
            "latest_observation": deepcopy(new_observation),
            "observation_history": observation_history,
        }


# ------------------------------------------
# Calculate final profiler timing
# ------------------------------------------

PROFILER_END_TIME = datetime.now(
    MELBOURNE_TIMEZONE
)

PROFILER_DURATION_SECONDS = (
    PROFILER_END_TIME - PROFILER_START_TIME
).total_seconds()

PROFILER_DURATION = str(
    PROFILER_END_TIME - PROFILER_START_TIME
)


# ------------------------------------------
# Determine final status
# ------------------------------------------

FINAL_RUN_STATUS = determine_run_status(
    files_processed=files_processed_successfully_count,
    files_failed=files_failed_count,
    files_with_drift=files_with_drift_count,
    files_with_proposal_variance=(
        files_with_proposed_schema_variance_count
    ),
)


# ------------------------------------------
# Update Pandas schema observations
# ------------------------------------------

update_py_schema_observations(
    registry=py_registry,
    current_observations=(
        observed_pandas_schemas_by_file
    ),
)


# ------------------------------------------
# Build shared run-history record
# ------------------------------------------

run_history_record = {
    "session_id": SESSION_ID,
    "start_time": PROFILER_START_TIME.isoformat(
        timespec="seconds"
    ),
    "end_time": PROFILER_END_TIME.isoformat(
        timespec="seconds"
    ),
    "duration_seconds": round(
        PROFILER_DURATION_SECONDS,
        3,
    ),
    "timezone": str(MELBOURNE_TIMEZONE),
    "status": FINAL_RUN_STATUS,
    "active_schema_status": master_registry[
        "active_schema_status"
    ],
    "active_schema_version": master_registry[
        "active_schema_version"
    ],
    "schema_comparison_mode": schema_comparison_mode,
    "profiling_mode": PROFILING_MODE,
    "sample_rows": EFFECTIVE_SAMPLE_ROWS,
    "files_discovered": files_discovered_count,
    "files_processed_successfully": (
        files_processed_successfully_count
    ),
    "files_failed": files_failed_count,
    "files_with_drift": files_with_drift_count,
    "files_with_proposed_schema_variance": (
        files_with_proposed_schema_variance_count
    ),
    "processed_files": deepcopy(processed_files),
    "failed_files": deepcopy(failed_files),
    "drift_details": deepcopy(
        drift_details_by_file
    ),
    "proposed_schema_variance_details": deepcopy(
        proposed_schema_variance_by_file
    ),
    "warning_count": len(run_warnings),
    "error_count": len(run_errors),
}


# ------------------------------------------
# Append registry run histories
# ------------------------------------------

master_registry["run_history"].append(
    deepcopy(run_history_record)
)

py_registry["run_history"].append(
    deepcopy(run_history_record)
)


# ------------------------------------------
# Persist registry files
# ------------------------------------------

try:
    save_json_atomically(
        file_path=MASTER_SCHEMA_PATH,
        data=master_registry,
    )

    log_event(
        message=(
            "MASTER_REGISTRY_SAVED: "
            f"path={MASTER_SCHEMA_PATH}; "
            f"status={FINAL_RUN_STATUS}."
        ),
        level="INFO",
    )

    save_json_atomically(
        file_path=PY_SCHEMA_PATH,
        data=py_registry,
    )

    log_event(
        message=(
            "PY_REGISTRY_SAVED: "
            f"path={PY_SCHEMA_PATH}; "
            f"files_with_observations="
            f"{len(py_registry['schemas_by_file'])}."
        ),
        level="INFO",
    )

except OSError as error:
    log_event(
        message=(
            "REGISTRY_PERSISTENCE_FAILED: "
            f"error_type={type(error).__name__}; "
            f"error={error}"
        ),
        level="CRITICAL",
    )

    raise RuntimeError(
        "Profiler results could not be persisted safely."
    ) from error


# ------------------------------------------
# Build final profiler summary
# ------------------------------------------

FINAL_RUN_SUMMARY = {
    "session_id": SESSION_ID,
    "start_time": PROFILER_START_TIME.isoformat(
        timespec="seconds"
    ),
    "end_time": PROFILER_END_TIME.isoformat(
        timespec="seconds"
    ),
    "duration": PROFILER_DURATION,
    "duration_seconds": round(
        PROFILER_DURATION_SECONDS,
        3,
    ),
    "timezone": str(MELBOURNE_TIMEZONE),
    "status": FINAL_RUN_STATUS,
    "active_schema_status": master_registry[
        "active_schema_status"
    ],
    "active_schema_version": master_registry[
        "active_schema_version"
    ],
    "proposed_schema_status": (
        master_registry.get(
            "proposed_schema_metadata"
        ) or {}
    ).get("status"),
    "proposal_created_this_run": (
        proposal_created_this_run
    ),
    "proposal_source_file": proposal_source_file,
    "schema_comparison_mode": schema_comparison_mode,
    "profiling_mode": PROFILING_MODE,
    "sample_rows": EFFECTIVE_SAMPLE_ROWS,
    "files_discovered": files_discovered_count,
    "files_processed_successfully": (
        files_processed_successfully_count
    ),
    "files_failed": files_failed_count,
    "files_with_drift": files_with_drift_count,
    "files_with_proposed_schema_variance": (
        files_with_proposed_schema_variance_count
    ),
    "processed_files": deepcopy(processed_files),
    "failed_files": deepcopy(failed_files),
    "errors": deepcopy(run_errors),
    "warnings": deepcopy(run_warnings),
    "drift_details": deepcopy(
        drift_details_by_file
    ),
    "proposed_schema_variance_details": deepcopy(
        proposed_schema_variance_by_file
    ),
}


# ------------------------------------------
# Record profiler session end
# ------------------------------------------

log_event(
    message=(
        "SESSION_END: CSV schema diagnostic profiler closed. "
        f"status={FINAL_RUN_STATUS}; "
        f"files_discovered={files_discovered_count}; "
        "files_processed="
        f"{files_processed_successfully_count}; "
        f"files_failed={files_failed_count}; "
        f"files_with_drift={files_with_drift_count}; "
        "files_with_proposed_schema_variance="
        f"{files_with_proposed_schema_variance_count}; "
        f"duration_seconds="
        f"{round(PROFILER_DURATION_SECONDS, 3)}."
    ),
    level="INFO",
)


# ------------------------------------------
# Display final run summary
# ------------------------------------------

print("\n========== Final Profiler Run Summary ==========\n")

print(
    f"Session ID                       : "
    f"{FINAL_RUN_SUMMARY['session_id']}"
)
print(
    f"Start time                       : "
    f"{FINAL_RUN_SUMMARY['start_time']}"
)
print(
    f"End time                         : "
    f"{FINAL_RUN_SUMMARY['end_time']}"
)
print(
    f"Duration                         : "
    f"{FINAL_RUN_SUMMARY['duration']}"
)
print(
    f"Timezone                         : "
    f"{FINAL_RUN_SUMMARY['timezone']}"
)
print(
    f"Status                           : "
    f"{FINAL_RUN_SUMMARY['status']}"
)
print(
    f"Active schema status             : "
    f"{FINAL_RUN_SUMMARY['active_schema_status']}"
)
print(
    f"Active schema version            : "
    f"{FINAL_RUN_SUMMARY['active_schema_version']}"
)
print(
    f"Proposed schema status           : "
    f"{FINAL_RUN_SUMMARY['proposed_schema_status']}"
)
print(
    f"Schema comparison mode           : "
    f"{FINAL_RUN_SUMMARY['schema_comparison_mode']}"
)
print(
    f"Profiling mode                   : "
    f"{FINAL_RUN_SUMMARY['profiling_mode']}"
)
print(
    f"Sample rows                      : "
    f"{FINAL_RUN_SUMMARY['sample_rows']}"
)
print(
    f"Files discovered                 : "
    f"{FINAL_RUN_SUMMARY['files_discovered']}"
)
print(
    f"Files processed successfully     : "
    f"{FINAL_RUN_SUMMARY['files_processed_successfully']}"
)
print(
    f"Files failed                     : "
    f"{FINAL_RUN_SUMMARY['files_failed']}"
)
print(
    f"Files with approved-schema drift : "
    f"{FINAL_RUN_SUMMARY['files_with_drift']}"
)
print(
    "Files with proposal variance     : "
    f"{FINAL_RUN_SUMMARY[
        'files_with_proposed_schema_variance'
    ]}"
)
print(
    f"Warnings                         : "
    f"{len(FINAL_RUN_SUMMARY['warnings'])}"
)
print(
    f"Errors                           : "
    f"{len(FINAL_RUN_SUMMARY['errors'])}"
)


# ------------------------------------------
# Display important diagnostic details
# ------------------------------------------

if FINAL_RUN_SUMMARY["failed_files"]:
    print("\nFailed files:")

    for failed_file in FINAL_RUN_SUMMARY["failed_files"]:
        print(
            f"  - {failed_file['file_name']}: "
            f"{failed_file['error_type']} — "
            f"{failed_file['error_message']}"
        )


if FINAL_RUN_SUMMARY["drift_details"]:
    print("\nApproved-schema drift details:")

    for file_name, differences in (
        FINAL_RUN_SUMMARY["drift_details"].items()
    ):
        print(f"\n  {file_name}")

        for column_name, detail in differences.items():
            print(
                f"    - {column_name}: "
                f"{detail['issue']}; "
                f"expected={detail['expected']}; "
                f"observed={detail['observed']}"
            )


if FINAL_RUN_SUMMARY[
    "proposed_schema_variance_details"
]:
    print("\nProposed-schema variance details:")

    for file_name, differences in (
        FINAL_RUN_SUMMARY[
            "proposed_schema_variance_details"
        ].items()
    ):
        print(f"\n  {file_name}")

        for column_name, detail in differences.items():
            print(
                f"    - {column_name}: "
                f"{detail['issue']}; "
                f"expected={detail['expected']}; "
                f"observed={detail['observed']}"
            )


print("\n" + "=" * 75)

if FINAL_RUN_STATUS == "SUCCESS":
    print("✓ Profiler completed successfully.")

elif FINAL_RUN_STATUS == "SCHEMA_DRIFT_DETECTED":
    print("⚠ Profiler completed with approved-schema drift.")

elif (
    FINAL_RUN_STATUS
    == "PROPOSED_SCHEMA_VARIANCE_DETECTED"
):
    print(
        "⚠ Profiler completed with variance against the "
        "proposed schema."
    )

elif FINAL_RUN_STATUS == "PARTIAL_FAILURE":
    print(
        "⚠ Profiler completed, but one or more files "
        "could not be processed."
    )

else:
    print("✗ Profiler failed to perform meaningful profiling.")

print("✓ Master schema registry updated.")
print("✓ Pandas schema registry updated.")
print("✓ Profiler session closed.")
print("=" * 75)